# Machine Learning Programming(PROG8245)

### LAB 2- Data Collection and Pre-processing

**Student Name**: Hasyashri Bhatt

**Student ID**  : 9028501

**Course**      : Machine Learning Programming-PROG8245


# Step 1: Hello, Data!

**Load raw CSV, display first 3 rows**

In [ ]:
# Importing Libraries to Load synthetic ecommerce dataset with 500 rows

import pandas as pd
import numpy as np


In [ ]:
# Load the dataset
df= pd.read_csv(r'C:\Users\habha\MLprogramming_PROG8245\Lab2_Ecommerce_Data_Collection_and_Preprocessing\data\enhanced_ecommerce_transactions.csv')
print("Dataset loaded successfully!")

In [ ]:
# Display first 3 rows
df.head(3)

# Step 2: Pick the Right Container

**I choose `class` instead of `dict` and `namedtuple` because it allows encapsulating behavior and state together. `Dictonary` lacks structure and `namedtuple` is mutable so limiting data updates.**

# Step 3: Transaction Class and OO data structure

**Implement Transaction class and use it t populate an object-oriented data structure**

used `copilot` for this below code

In [ ]:
class Transaction:
    """Class to represent a single transaction in the ecommerce dataset."""
    def __init__(self, date, customer_id, product, price, quantity, coupon_code,
                 shipping_city, payment_method, order_status, delivery_date, customer_email):
        self.date = date
        self.customer_id = customer_id
        self.product = product
        self.price = price
        self.quantity = quantity
        self.coupon_code = coupon_code
        self.shipping_city = shipping_city
        self.payment_method = payment_method
        self.order_status = order_status
        self.delivery_date = delivery_date
        self.customer_email = customer_email

    def __repr__(self):
        return (f"Transaction({self.date}, {self.customer_id}, {self.product}, "
                f"{self.price}, {self.quantity}, {self.coupon_code}, {self.shipping_city}, "
                f"{self.payment_method}, {self.order_status}, {self.delivery_date}, {self.customer_email})")

# Populate a list of Transaction objects from the dataframe
""" This code iterates over each row in the DataFrame and creates a Transaction object for each row, storing them in a list."""
transactions = [
    Transaction(
        row['date'],
        row['customer_id'],
        row['product'],
        row['price'],
        row['quantity'],
        row['coupon_code'],
        row['shipping_city'],
        row['payment_method'],
        row['order_status'],
        row['delivery_date'],
        row['customer_email']
    )
    for _, row in df.iterrows()
]

# Example: print the first 3 transactions
for t in transactions[:3]:
    print(t)

# Step 4: Bulk Loader

**load_transactions() returning list ? type-hinted**
used `copilot` for this code generation

In [ ]:
""" This code defines a class `Transaction` to represent a single transaction in an ecommerce dataset, initializes it with various attributes, and provides a method to return a string representation of the object. It then loads transactions from a DataFrame into a list of `Transaction` objects."""
from typing import List

""" Function to load transactions from a DataFrame and return a list of Transaction objects."""
def load_transactions(df: pd.DataFrame) -> List[Transaction]:
    """Load transactions from a DataFrame and return a list of Transaction objects."""
    return [
        Transaction(
            row['date'],
            row['customer_id'],
            row['product'],
            row['price'],
            row['quantity'],
            row['coupon_code'],
            row['shipping_city'],
            row['payment_method'],
            row['order_status'],
            row['delivery_date'],
            row['customer_email']
        )
        for _, row in df.iterrows()
    ]

# Example usage:
transactions = load_transactions(df)
# Print the transactions to verify in table format
for t in transactions[:3]:
    print(t)

# Step 5: Quick Profiling

**Min/mean/max price, unique city count (set)**

used `copilot` for the code generation

In [ ]:
# Quick profiling: min/mean/max price and unique shipping city count
min_price = df['price'].min()
mean_price = df['price'].mean()
max_price = df['price'].max()
unique_cities = set(df['shipping_city'])

print(f"Min price: {min_price}")
print(f"Mean price: {mean_price:.2f}")
print(f"Max price: {max_price}")
print(f"Unique shipping cities ({len(unique_cities)}): {unique_cities}")

# Step 6:	Spot the Grime

**Identify at least three dirty data cases**

used copilot for the code generation during the class

In [ ]:
# Identify at least three dirty data cases in the DataFrame

# 1. Missing values in any column
missing_values = df.isnull().sum()
print("Missing values per column:\n", missing_values)

# 2. Duplicate rows
duplicate_rows = df[df.duplicated()]
print(f"Number of duplicate rows: {len(duplicate_rows)}")

# 3. Invalid prices (e.g., price <= 0)
invalid_prices = df[df['price'] <= 0]
print(f"Number of transactions with invalid price: {len(invalid_prices)}")

# 4. Invalid quantities (e.g., quantity <= 0)
invalid_quantities = df[df['quantity'] <= 0]
print(f"Number of transactions with invalid quantity: {len(invalid_quantities)}")

# Step 7: 	Cleaning Rules

**Execute fixes inside clean(); show ?before/after? counts**

used copilot for code

In [ ]:
def clean(transactions: list) -> list:
    """
    Remove transactions with invalid price/quantity or missing fields.
    Returns a new list of cleaned transactions.
    """
    cleaned = []
    for t in transactions:
        # Check for valid price and quantity
        if t.price is None or t.price <= 0:
            continue
        if t.quantity is None or t.quantity <= 0:
            continue
        # Check for missing required fields
        required_fields = [
            t.date, t.customer_id, t.product, t.coupon_code,
            t.shipping_city, t.payment_method, t.order_status,
            t.delivery_date, t.customer_email
        ]
        if any(field is None or field == "" for field in required_fields):
            continue
        cleaned.append(t)
    return cleaned

before_cleaning = len(transactions)
cleaned_transactions = clean(transactions)
after_cleaning = len(cleaned_transactions)

print(f"Transactions before cleaning: {before_cleaning}")
print(f"Transactions after cleaning: {after_cleaning}")

# Step 8: Transformations

**	For example: Parse coupon_code ? numeric discount (others apply)**

used copilot for coding

In [ ]:
# Transformation: Parse coupon_code to numeric discount and add as attribute

def parse_discount(coupon_code: str) -> float:
    """
    Convert coupon_code to a numeric discount percentage.
    Example rules (customize as needed):
      - 'SAVE10' or 'WELCOME10' -> 10%
      - 'WELCOME20' -> 20%
      - 'BUY1GET1' -> 50% (assuming buy one get one free)
      - 'FREESHIP' -> 0% (no price discount)
      - Unknown/empty -> 0%
    """
    if not coupon_code or not isinstance(coupon_code, str):
        return 0.0
    code = coupon_code.upper()
    if '10' in code:
        return 10.0
    if '20' in code:
        return 20.0
    if 'BUY1GET1' in code:
        return 50.0
    if 'FREESHIP' in code:
        return 0.0
    return 0.0

# Add discount attribute to each cleaned transaction
for t in cleaned_transactions:
    t.discount = parse_discount(t.coupon_code)

# Example: print first 5 transactions with their discount
for t in cleaned_transactions[:5]:
    print(f"{t} | Discount: {t.discount}%")

# Step 9: Feature Engineering
**For example: Add days_since_purchase**

used copilot for code


In [ ]:
# Feature Engineering: Add 'is_high_value_order' column (True if total_value > $1000)

import datetime

# Set today's date
today = pd.Timestamp(datetime.date.today())

# Convert cleaned_transactions (list of Transaction objects) to DataFrame
cleaned_df = pd.DataFrame([t.__dict__ for t in cleaned_transactions])

# Ensure 'date' column is datetime
cleaned_df['date'] = pd.to_datetime(cleaned_df['date'])

# Add 'days_since_purchase'
cleaned_df['days_since_purchase'] = (today - cleaned_df['date']).dt.days

# Add 'total_value' (price * quantity * (1 - discount/100))
cleaned_df['total_value'] = cleaned_df['price'] * cleaned_df['quantity'] * (1 - cleaned_df['discount'] / 100)

# Add a boolean column indicating if the order is high value
cleaned_df['is_high_value_order'] = cleaned_df['total_value'] > 1000

# Show first 5 rows with the new columns
cleaned_df.head()

# Step 10: Mini-Aggregation
Used co pilot

In [ ]:
# Calculate average order value per payment_method using pandas groupby
avg_order_value_by_payment = cleaned_df.groupby('payment_method')['total_value'].mean().to_dict()
print("Average order value per payment method:")
for method, avg_value in avg_order_value_by_payment.items():
    print(f"{method}: ${avg_value:.2f}")

# Step 11:Serialization Checkpoint

**Save cleaned data to JSON and Parquet**

Used copilot

In [ ]:

import os


# Example cleaned DataFrame (replace this with your actual cleaned data)
df = pd.read_csv(r'C:\Users\habha\MLprogramming_PROG8245\Lab2_Ecommerce_Data_Collection_and_Preprocessing\data\enhanced_ecommerce_transactions.csv')
# Clean the DataFrame (if not already done)

# Define the target directory (relative to the current working directory)
target_directory = "../data"

# Ensure the directory exists
# os.makedirs(target_directory, exist_ok=True)
# save clean csv file
cleaned_csv_path = os.path.join(target_directory, "cleaned_data.csv")
df.to_csv(cleaned_csv_path, index=False)
print(f"Data saved to CSV at {cleaned_csv_path}")

# Save as JSON
cleaned_json_path = os.path.join(target_directory, "cleaned_data.json")
df.to_json(cleaned_json_path, orient='records', lines=False)
print(f"Data saved to JSON at {cleaned_json_path}")

# Save as Parquet
cleaned_parquet_path = os.path.join(target_directory, "cleaned_data.parquet")
df.to_parquet(cleaned_parquet_path, index=False)
print(f"Data saved to Parquet at {cleaned_parquet_path}")

# Step 12: Soft Interview Reflection	
**Using Object-Oriented Programming made data management easier by grouping related data and operations into single structure. The Transaction class allowed me to organize data consistently,simplify debugging, and make future feature enhancements more straightforward.**

# Data-Dictionary Section

**Merge field definitions from the primary CSV header and the secondary metadata source.**
**Present as a tidy Markdown table including the new columns, for example: Field, Type, Description, Source.**
used co-pilot

In [ ]:
import json

# Load metadata
with open("../data/secondary_metadata.json", "r") as f:
    metadata = json.load(f)

# Extract product catalog and city lookups
product_catalog = {entry["product"]: {"category": entry["category"], "brand": entry["brand"]} for entry in metadata["product_catalog"]}
city_regions = {entry["city"]: entry["region"] for entry in metadata["city_lookups"]}

# Define primary fields
primary_fields = [
    {"name": "date", "type": "Date", "description": "Transaction date"},
    {"name": "customer_id", "type": "String", "description": "Unique identifier for customers"},
    {"name": "customer_email", "type": "String", "description": "Email address of the customer"},
    {"name": "product", "type": "String", "description": "Name of the purchased product"},
    {"name": "price", "type": "Float", "description": "Price of the product"},
    {"name": "quantity", "type": "Integer", "description": "Quantity of the product purchased"},
    {"name": "coupon_code", "type": "String", "description": "Applied discount coupon code"},
    {"name": "shipping_city", "type": "String", "description": "City to which the order was shipped"},
    {"name": "delivery_date", "type": "Date", "description": "Estimated delivery date for the order"},
    {"name": "discount_percent", "type": "Float", "description": "Discount derived from coupon code"},
    {"name": "days_since_purchase", "type": "Integer", "description": "Days elapsed since the purchase"},
    {"name": "total_value", "type": "Float", "description": "Total value of the order"},
    {"name": "is_high_value_order", "type": "Boolean", "description": "Whether the order is a high-value transaction"}
]

# Define secondary metadata fields
metadata_fields = {
    "region": {"type": "String", "description": "Region of the shipping city"},
    "category": {"type": "String", "description": "Product category"},
    "brand": {"type": "String", "description": "Brand of the product"}
}

# Merge Field Definitions
def merge_field_definitions(primary_fields, metadata_fields):
    combined_fields = []
    for field in primary_fields:
        combined_fields.append({
            "Field": field["name"],
            "Type": field["type"],
            "Description": field["description"],
            "Source": "Primary CSV"
        })

    for field, details in metadata_fields.items():
        combined_fields.append({
            "Field": field,
            "Type": details["type"],
            "Description": details["description"],
            "Source": "Metadata"
        })

    return combined_fields

# Merge definitions
merged_definitions = merge_field_definitions(primary_fields, metadata_fields)

# Print as Markdown Table
print("| Field               | Type     | Description                                       | Source                 |")
print("|---------------------|----------|---------------------------------------------------|------------------------|")
for field in merged_definitions:
    print(f"| {field['Field']:20} | {field['Type']:8} | {field['Description']:50} | {field['Source']:20} |")
